# Agrim — run the full research study in Google Colab

This notebook reproduces every number shown on the website: it downloads the Open University
Learning Analytics Dataset (OULAD) from the UCI repository, verifies its SHA-256, builds the
point-in-time features, trains and compares all models, and writes the dashboard data, model
files, figures and a Markdown results report.

**Runtime → Change runtime type → CPU is enough.** The full study takes about 35–60 minutes; tick
`QUICK = True` for a 2-minute smoke run.

In [ ]:
REPO_URL = "https://github.com/abhinavaryan/academic-risk-prediction.git"  # change if you fork
QUICK = False

!git clone --depth 1 $REPO_URL agrim
%cd agrim
!pip install -q -r requirements-research.txt

## 1. Download and verify the dataset

In [ ]:
!python -m riskml.oulad.download --dest data/oulad
!ls -la data/oulad

## 2. Look at the features for one prediction day

In [ ]:
from pathlib import Path
from riskml.oulad.load import load_tables
from riskml.oulad.features import build_features

tables = load_tables(Path("data/oulad"), cache_dir=Path("data/cache"))
ff = build_features(tables, cutoff=60)
print(ff.X.shape, "at-risk rate:", round(ff.y.mean(), 3))
ff.X.describe().T.head(20)

## 3. Run the study

In [ ]:
flags = "--quick --out-root build/quick" if QUICK else ""
!python -m riskml.research.pipeline --data-dir data/oulad $flags

## 4. Results

In [ ]:
from IPython.display import Image, Markdown, display
root = Path("build/quick") if QUICK else Path(".")
report = root / "reports" / "results.md"
if report.exists():
    display(Markdown(report.read_text()))
for png in sorted((root / "reports" / "figures").glob("*.png")):
    display(Image(filename=str(png), width=640))

## 5. Use the new results in the website

Download `artifacts/`, `web/src/generated/` and `reports/` (Files panel → right-click → Download)
and copy them into your repository, or zip them:

In [ ]:
!zip -qr results.zip artifacts web/src/generated reports
from google.colab import files  # noqa
files.download("results.zip")